# create clear sky profile for blsn detection algorithm

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import importlib
from scipy.ndimage import median_filter
import glob
from windrose import WindroseAxes
import cmcrameri as cmc
from cmcrameri import cm
from scipy.stats import linregress
import sys
import os
from pathlib import Path
import pickle

###  import slowdata

In [ ]:
# import slowdata
pkl_2024 = 'D:\data\slowdata_cleaned\slowdata_cleaned_2024.pkl'
pkl_2025 = 'D:\data\slowdata_cleaned\slowdata_cleaned_2025.pkl'
slowdata_cleaned_2024 = pd.read_pickle(pkl_2024)
slowdata_cleaned_2025 = pd.read_pickle(pkl_2025)

slowdata_cleaned = pd.concat([slowdata_cleaned_2024, slowdata_cleaned_2025])

### select clear sky days
note: here we select them using sw radiation and wind speed data (differently from Gossart et al. (2013), where they select them manually looking at backscatter data). Then, to uniform with their findings, we will select instead of the 99th percentile profile, a percentile for which the lowest usable bin backscatter is the same they found at PE (21 sr-1 km-1 10-5)

In [ ]:
# --- Detect clear-sky + calm days from SW radiation ---
# Uses 5-min SW smoothness (rolling coefficient of variation) and wind speed
# to identify days suitable for building the ceilometer clear-sky baseline.

sw = slowdata_cleaned['SWdown1']
ws = slowdata_cleaned['WS2_Avg']

# 5-min resamples
sw_5min = sw.resample('5min').mean()
ws_5min = ws.resample('5min').mean()

# Daytime windows
daytime_5min = sw_5min > 50  # W/m²

# SW smoothness: 15-min rolling coefficient of variation on 1-min data, averaged to 5 min
sw_roll_std  = sw.rolling('15min', center=True).std()
sw_roll_mean = sw.rolling('15min', center=True).mean()
sw_cv        = (sw_roll_std / sw_roll_mean.where(sw_roll_mean > 50)).resample('5min').mean()

# Per-5-min flags
clear_5min      = daytime_5min & (sw_cv < 0.05)
calm_5min       = ws_5min < 6.0
clear_calm_5min = clear_5min & calm_5min

# Daily summaries
daily_sw_peak        = sw.resample('D').max()
daily_daytime_n      = daytime_5min.resample('D').sum()
daily_clear_calm_n   = clear_calm_5min.resample('D').sum()
daily_clear_fraction = daily_clear_calm_n / daily_daytime_n.replace(0, np.nan)

# --- Thresholds (adjust to your site/season) ---
SW_PEAK_MIN    = 200   # W/m²  minimum daily peak SW to qualify
CLEAR_FRAC_MIN = 0.60  # minimum fraction of daytime 5-min windows that are clear+calm

clear_sky_days = daily_clear_fraction.index[
    (daily_sw_peak        >= SW_PEAK_MIN) &
    (daily_clear_fraction >= CLEAR_FRAC_MIN)
]

print(f'Found {len(clear_sky_days)} clear-sky + calm days:')
for d in clear_sky_days:
    print(f'  {d.date()}  peak SW = {daily_sw_peak[d]:6.0f} W/m²  '
          f'clear+calm fraction = {daily_clear_fraction[d]:.0%}')

In [ ]:
# --- Plot all detected clear-sky + calm days: daily SW curves overlaid ---
import matplotlib.cm as mplcm
import matplotlib.colors as mcolors

fig, ax = plt.subplots(figsize=(10, 5))

if len(clear_sky_days) == 0:
    ax.text(0.5, 0.5, 'No clear-sky days found — lower SW_PEAK_MIN or CLEAR_FRAC_MIN',
            ha='center', va='center', transform=ax.transAxes)
else:
    cmap = mplcm.viridis
    norm = mcolors.Normalize(vmin=0, vmax=max(len(clear_sky_days) - 1, 1))
    for i, day in enumerate(clear_sky_days):
        day_sw = sw[sw.index.normalize() == day]
        hours  = (day_sw.index - day_sw.index.normalize()) / pd.Timedelta('1h')
        ax.plot(hours, day_sw.values, color=cmap(norm(i)), alpha=0.8, lw=1,
                label=str(day.date()))

ax.set_xlabel('Hour of day [UTC]')
ax.set_ylabel('Downwelling SW [W/m²]')
ax.set_title(f'Clear-sky + calm days (n = {len(clear_sky_days)}) — downwelling SW radiation')
ax.set_xlim(0, 24)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### select percentile --> obtain profile

In [ ]:
#select profile for which bin1 = 21!!!!

# --- Build clear-sky ceilometer profile with backscatter = 21 at bin 1 ---
# For each detected clear-sky day, loads the corresponding ceilometer file,
# filters timesteps to wind < 6 m/s, then computes percentile bin-by-bin.
# Uses memory-efficient streaming: processes day-by-day, bin-by-bin.

from pathlib import Path
import warnings
import xarray as xr

CEIL_ROOTS = [Path('D:/data/ceilo/2024')]
WIND_MAX   = 6.0  # m/s — per-profile wind filter

def _build_time_index(ds):
    """Reconstruct DatetimeIndex from base_time (epoch s) + time (decimal hours)."""
    if 'base_time' in ds.variables and 'time' in ds.variables:
        base_epoch    = float(np.array(ds['base_time']).squeeze())
        base_dt       = pd.to_datetime(base_epoch, unit='s', utc=True).tz_localize(None)
        decimal_hours = np.array(ds['time'].values, dtype=float)
        return pd.DatetimeIndex(base_dt + pd.to_timedelta(decimal_hours, unit='h'))
    return pd.DatetimeIndex(pd.to_datetime(ds['time'].values, errors='coerce'))

# --- FIRST PASS: Find target percentile from bin-1 values ---
print('FIRST PASS: Collecting bin-1 values to find target percentile...')
bin_1_values_list = []
n_bins = None
total_timesteps_kept = 0

for day in clear_sky_days:
    pattern = f'peaceilCL31.b1.{day.strftime("%Y%m%d")}*.nc'
    matches = []
    for ceil_root in CEIL_ROOTS:
        matches.extend(sorted((ceil_root / day.strftime('%Y%m')).glob(pattern)))
    if not matches:
        print(f'{day.date()}: no ceilometer file found — skipping')
        continue

    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            ds = xr.open_dataset(matches[0], decode_times=False)

        time_index = _build_time_index(ds)
        bsc_raw    = np.array(ds['backscatter'])
        ds.close()

        bsc = bsc_raw.T
        if n_bins is None:
            n_bins = bsc.shape[1]

        day_wind  = ws.reindex(time_index, method='nearest', tolerance=pd.Timedelta('2min'))
        calm_mask = (day_wind.values < WIND_MAX) & (~np.isnan(day_wind.values))
        kept = bsc[calm_mask]
        
        # Collect bin-1 as Python list (more memory efficient)
        bin_1_vals = kept[:, 1]
        bin_1_values_list.extend(bin_1_vals[~np.isnan(bin_1_vals)].tolist())
        
        total_timesteps_kept += calm_mask.sum()
        print(f'{day.date()}: {calm_mask.sum():4d} / {len(calm_mask)} timesteps kept')
    except Exception as e:
        print(f'{day.date()}: error — {e}')

if not bin_1_values_list:
    raise RuntimeError('No qualifying profiles collected. Check CEIL_ROOT and clear_sky_days.')

# Compute target percentile from list
target_percentile = np.percentile(bin_1_values_list, np.searchsorted(np.sort(bin_1_values_list), 21.0) / len(bin_1_values_list) * 100)
print(f'Target percentile for bin-1 = 21: {target_percentile:.2f} (from {total_timesteps_kept} profiles)')

In [ ]:
# Use the target percentile from the first pass, ensuring it's at least 95%
PERCENTILE = target_percentile
print(f'Using PERCENTILE = {PERCENTILE:.6f}% (target_percentile = {target_percentile:.6f}%)')
from pathlib import Path
import warnings
import xarray as xr

CEIL_ROOT  = Path(r'D:\data\ceilo\2024'), Path
WIND_MAX   = 6.0  # m/s — per-profile wind filter

def _build_time_index(ds):
    """Reconstruct DatetimeIndex from base_time (epoch s) + time (decimal hours)."""
    if 'base_time' in ds.variables and 'time' in ds.variables:
        base_epoch    = float(np.array(ds['base_time']).squeeze())
        base_dt       = pd.to_datetime(base_epoch, unit='s', utc=True).tz_localize(None)
        decimal_hours = np.array(ds['time'].values, dtype=float)
        return pd.DatetimeIndex(base_dt + pd.to_timedelta(decimal_hours, unit='h'))
    return pd.DatetimeIndex(pd.to_datetime(ds['time'].values, errors='coerce'))

all_profiles = []
import time

for idx, day in enumerate(clear_sky_days):
    print(f'\n[{idx+1}/{len(clear_sky_days)}] Processing {day.date()}...')
    print(f'  → Searching for ceilometer file...')
    
    t0 = time.time()
    pattern = f'peaceilCL31.b1.{day.strftime("%Y%m%d")}*.nc'
    for ceil_root in CEIL_ROOTS:
        matches = sorted((ceil_root / day.strftime('%Y%m')).glob(pattern))
        if not matches:
            print(f'  → NO CEILOMETER FILE FOUND — skipping')
        continue

    try:
        print(f'  → Opening {matches[0].name}...')
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            ds = xr.open_dataset(matches[0], decode_times=False)

        print(f'  → Building time index...')
        time_index = _build_time_index(ds)
        print(f'  → Loading backscatter array...')
        bsc_raw    = np.array(ds['backscatter'], dtype=np.float32)   # Use float32 to save memory
        print(f'    Backscatter shape: {bsc_raw.shape}')
        ds.close()

        # Transpose to (n_time, n_range) for indexing along the time axis
        bsc = bsc_raw.T
        print(f'  → Transposed shape: {bsc.shape}')

        # Match ceilometer timestamps to nearest 1-min wind observation
        print(f'  → Matching wind data...')
        day_wind  = ws.reindex(time_index, method='nearest', tolerance=pd.Timedelta('2min'))
        calm_mask = (day_wind.values < WIND_MAX) & (~np.isnan(day_wind.values))
        print(f'  → Wind filtering: {calm_mask.sum()} / {len(calm_mask)} kept')

        kept = bsc[calm_mask, :].astype(np.float32)   # (n_calm, n_range), keep as float32
        all_profiles.append(kept)
        elapsed = time.time() - t0
        print(f'  ✓ DONE in {elapsed:.1f}s')
    except Exception as e:
        print(f'  ✗ ERROR: {e}')

if not all_profiles:
    raise RuntimeError('No qualifying profiles collected. Check CEIL_ROOT and clear_sky_days.')

# Instead of vstack, calculate percentile bin-by-bin to save memory
# For each bin, collect all values across all days, then percentile
print('\nCalculating percentiles bin-by-bin (memory-efficient)...')
n_bins = all_profiles[0].shape[1] if len(all_profiles) > 0 else 0
clear_profile = np.zeros(n_bins, dtype=np.float32)

for bin_idx in range(n_bins):
    if bin_idx % 100 == 0:
        print(f'  Processing bin {bin_idx}/{n_bins}...')
    # Collect all values for this bin across all days
    bin_values = np.concatenate([profile[:, bin_idx] for profile in all_profiles])
    # Calculate percentile for this bin
    clear_profile[bin_idx] = np.nanpercentile(bin_values, PERCENTILE)

# Count total timesteps
total_timesteps = sum(p.shape[0] for p in all_profiles)
print(f'\nTotal profiles: {total_timesteps} timesteps across {len(all_profiles)} days')
print(f'Profile length: {len(clear_profile)} range bins')

out_file = Path('./clear_sky_profile_bin1_21.csv')
pd.DataFrame([clear_profile]).to_csv(out_file, header=False, index=False)
print(f'Saved: {out_file}')

In [ ]:
out_file = Path('./clear_sky_profile_bin1_21.csv')
pd.DataFrame([clear_profile]).to_csv(out_file, header=False, index=False)
print(f'Saved: {out_file}')

In [ ]:

# --- Plot the 95th-percentile clear-sky ceilometer profile ---
csv_path = Path('./clear_sky_profile_bin1_21.csv')

if 'clear_profile' in globals() and clear_profile is not None:
    profile = np.asarray(clear_profile, dtype=float)
elif csv_path.exists():
    profile = pd.read_csv(csv_path, header=None).iloc[0].to_numpy(dtype=float)
else:
    raise FileNotFoundError(f'No in-memory clear_profile and CSV not found at {csv_path}')

y       = np.arange(len(profile))
y_label = 'Bin index'

fig, ax = plt.subplots(figsize=(6, 8))
ax.plot(profile, y, lw=2, color='tab:blue')
ax.set_xlabel('Backscatter')
ax.set_ylabel(y_label)
ax.set_title(f'Clear-sky ceilometer profile')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()